In [65]:
%pip install torch
%pip install scanpy
%pip install numpy
%pip install pandas
%pip install scikit-learn


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [66]:
# Install dependencies
import torch
import pandas as pd
import scanpy
import numpy as np
import pandas
import sklearn
import subprocess
import os

## DeepSEM expects columns to correspond to genes and rows to correspond to cells, so the data frame containing expression values must be transposed before it can be used as input to DeepSEM.

In [67]:
# Read in the expression data
df_expression = pd.read_csv("CHOOSE-multiome-wt-log-norm.csv.gz", compression="gzip")

# Keep track of the original number of rows, columns, and sum of data points to ensure no data is dropped after transposing the data frame
orig_genes = df_expression.shape[0]
orig_cells = df_expression.iloc[:, 1:].shape[1]
orig_sum = np.sum(df_expression.iloc[:, 1:].values)

# Set gene names as the row labels and drop them from the data frame so they are not treated as data
df_expression.index = df_expression.iloc[:, 0]
df_expression = df_expression.drop(columns=[df_expression.columns[0]])

# Prevent pandas from assigning a title to the row labels so it doesn't get added to the CSV file later on
df_expression.index.name = None

# Transpose the data frame
df_transposed = df_expression.T

# Validation: compare the row/col counts
if (orig_genes == df_transposed.shape[1]) and (orig_cells == df_transposed.shape[0]):
    print("Rows/cols preserved.")
else:
    print("Rows/cols do not match.")

# Validation: compare the sums
if orig_sum == np.sum(df_transposed.values):
    print("Numerical data preserved.")
else:
    print("Numerical values are missing.")

# Write the transposed data frame to a CSV file to be used as input to DeepSEM
deepsem_input_file = 'input_data.csv'
df_transposed.to_csv(deepsem_input_file, sep='\t', header=True, index=True)

Rows/cols preserved.
Numerical data preserved.


## Since we are building a cell type specific GRN, DeepSEM needs the celltype_jf data from the metadata file.

In [68]:
df_meta = pd.read_csv("CHOOSE-multiome-wt-metadata.csv", header=None, skiprows=1)
original_headers = pd.read_csv("CHOOSE-multiome-wt-metadata.csv", nrows=0).columns.tolist()
corrected_headers = ['cell_id'] + original_headers
df_meta.columns = corrected_headers[:df_meta.shape[1]]
df_meta = df_meta.set_index('cell_id')
df_meta.index.name = None
# Proceed with the rest of the alignment to the transposed expression matrix
df_meta_aligned = df_meta.loc[df_transposed.index]
df_cell_types = df_meta_aligned[['celltype_jf']]

## Run DeepSEM with the transposed expression data.

In [69]:
output_dir = 'DeepSEM_Results/'
os.makedirs(output_dir, exist_ok=True)

unique_cell_types = df_cell_types['celltype_jf'].unique()

# Loop through each cell type and run DeepSEM
for c_type in unique_cell_types:
    print(f"\n--- Processing Cell Type: {c_type} ---")

    # 1. Isolate the cell IDs for this specific type
    cells_in_type = df_cell_types[df_cell_types['celltype_jf'] == c_type].index

    # 2. Subset the expression matrix to ONLY include these cells
    df_subset = df_transposed.loc[cells_in_type]

    # 3. Save the subset to a temporary file
    # (Replacing spaces/slashes in cell type names just in case to prevent file path errors)
    # 3. Save the subset to a standard CSV file
    safe_c_type = str(c_type).replace(" ", "_").replace("/", "_")

    # CHANGE 1: Switch from .tsv to .csv
    subset_file = f"deepsem_input_{safe_c_type}.csv"

    # I added the gene count to the print statement just so we can verify the 15685 vs 15694 discrepancy
    print(f"Saving subset matrix ({df_subset.shape[0]} cells, {df_subset.shape[1]} genes) to {subset_file}...")

    # CHANGE 2: Remove the sep='\t' and index_label="" hacks.
    # Just save it as a completely standard CSV! Scanpy natively loves this format.
    df_subset.to_csv(subset_file, header=True, index=True)

    # 4. Define the specific output name for this run
    save_prefix = os.path.join(output_dir, f"DeepSEM_out_{safe_c_type}")

    # 5. Build the CORRECTED command
    deepsem_command = [
        "python", "DeepSEM-master/main.py",
        "--task", "non_celltype_GRN",
        "--data_file", subset_file,         # Now points to the .csv
        "--save_name", save_prefix,
        "--setting", "test",
        "--n_epochs", "120"
    ]

    # 6. Run the model
    try:
        subprocess.run(deepsem_command, check=True)
        print(f"✅ Successfully generated network for {c_type}.")
    except subprocess.CalledProcessError as e:
        print(f"❌ Error processing {c_type}: {e}")

print("\nAll cell types processed!")


--- Processing Cell Type: ge_in ---
Saving subset matrix (126 cells, 15685 genes) to deepsem_input_ge_in.csv...
dir exist
dir exist


/home/christina/PycharmProjects/TCSS-478-588-Project/DeepSEM-master/src/DeepSEM_cell_type_test_non_specific_GRN_model.py:37: RuntimeWarning: invalid value encountered in divide
  data_values = (data_values - data_values.mean(0)) / (data_values.std(0))
Traceback (most recent call last):
  File "/home/christina/PycharmProjects/TCSS-478-588-Project/DeepSEM-master/main.py", line 53, in <module>
    model.train_model()
  File "/home/christina/PycharmProjects/TCSS-478-588-Project/DeepSEM-master/src/DeepSEM_cell_type_test_non_specific_GRN_model.py", line 51, in train_model
    vae = VAE_EAD(adj_A_init, 1, opt.n_hidden, opt.K).float().cuda()
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/christina/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1093, in cuda
    return self._apply(lambda t: t.cuda(device))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/christina/.venv/lib/python3.12/site-packages/torch/nn/modules/module.p

❌ Error processing ge_in: Command '['python', 'DeepSEM-master/main.py', '--task', 'non_celltype_GRN', '--data_file', 'deepsem_input_ge_in.csv', '--save_name', 'DeepSEM_Results/DeepSEM_out_ge_in', '--setting', 'test', '--n_epochs', '120']' returned non-zero exit status 1.

--- Processing Cell Type: ctx_ex ---
Saving subset matrix (185 cells, 15685 genes) to deepsem_input_ctx_ex.csv...
dir exist
dir exist


/home/christina/PycharmProjects/TCSS-478-588-Project/DeepSEM-master/src/DeepSEM_cell_type_test_non_specific_GRN_model.py:37: RuntimeWarning: invalid value encountered in divide
  data_values = (data_values - data_values.mean(0)) / (data_values.std(0))
Traceback (most recent call last):
  File "/home/christina/PycharmProjects/TCSS-478-588-Project/DeepSEM-master/main.py", line 53, in <module>
    model.train_model()
  File "/home/christina/PycharmProjects/TCSS-478-588-Project/DeepSEM-master/src/DeepSEM_cell_type_test_non_specific_GRN_model.py", line 51, in train_model
    vae = VAE_EAD(adj_A_init, 1, opt.n_hidden, opt.K).float().cuda()
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/christina/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1093, in cuda
    return self._apply(lambda t: t.cuda(device))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/christina/.venv/lib/python3.12/site-packages/torch/nn/modules/module.p

❌ Error processing ctx_ex: Command '['python', 'DeepSEM-master/main.py', '--task', 'non_celltype_GRN', '--data_file', 'deepsem_input_ctx_ex.csv', '--save_name', 'DeepSEM_Results/DeepSEM_out_ctx_ex', '--setting', 'test', '--n_epochs', '120']' returned non-zero exit status 1.

--- Processing Cell Type: ge_npc ---
Saving subset matrix (240 cells, 15685 genes) to deepsem_input_ge_npc.csv...
dir exist
dir exist


...

❌ Error processing ge_npc: Command '['python', 'DeepSEM-master/main.py', '--task', 'non_celltype_GRN', '--data_file', 'deepsem_input_ge_npc.csv', '--save_name', 'DeepSEM_Results/DeepSEM_out_ge_npc', '--setting', 'test', '--n_epochs', '120']' returned non-zero exit status 1.

--- Processing Cell Type: ctx_npc ---
Saving subset matrix (81 cells, 15685 genes) to deepsem_input_ctx_npc.csv...
dir exist
dir exist


/home/christina/PycharmProjects/TCSS-478-588-Project/DeepSEM-master/src/DeepSEM_cell_type_test_non_specific_GRN_model.py:37: RuntimeWarning: invalid value encountered in divide
  data_values = (data_values - data_values.mean(0)) / (data_values.std(0))
Traceback (most recent call last):
  File "/home/christina/PycharmProjects/TCSS-478-588-Project/DeepSEM-master/main.py", line 53, in <module>
    model.train_model()
  File "/home/christina/PycharmProjects/TCSS-478-588-Project/DeepSEM-master/src/DeepSEM_cell_type_test_non_specific_GRN_model.py", line 51, in train_model
    vae = VAE_EAD(adj_A_init, 1, opt.n_hidden, opt.K).float().cuda()
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/christina/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1093, in cuda
    return self._apply(lambda t: t.cuda(device))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/christina/.venv/lib/python3.12/site-packages/torch/nn/modules/module.p

❌ Error processing ctx_npc: Command '['python', 'DeepSEM-master/main.py', '--task', 'non_celltype_GRN', '--data_file', 'deepsem_input_ctx_npc.csv', '--save_name', 'DeepSEM_Results/DeepSEM_out_ctx_npc', '--setting', 'test', '--n_epochs', '120']' returned non-zero exit status 1.

--- Processing Cell Type: ctx_ip ---
Saving subset matrix (100 cells, 15685 genes) to deepsem_input_ctx_ip.csv...
dir exist
dir exist


/home/christina/PycharmProjects/TCSS-478-588-Project/DeepSEM-master/src/DeepSEM_cell_type_test_non_specific_GRN_model.py:37: RuntimeWarning: invalid value encountered in divide
  data_values = (data_values - data_values.mean(0)) / (data_values.std(0))
Traceback (most recent call last):
  File "/home/christina/PycharmProjects/TCSS-478-588-Project/DeepSEM-master/main.py", line 53, in <module>
    model.train_model()
  File "/home/christina/PycharmProjects/TCSS-478-588-Project/DeepSEM-master/src/DeepSEM_cell_type_test_non_specific_GRN_model.py", line 51, in train_model
    vae = VAE_EAD(adj_A_init, 1, opt.n_hidden, opt.K).float().cuda()
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/christina/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1093, in cuda
    return self._apply(lambda t: t.cuda(device))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/christina/.venv/lib/python3.12/site-packages/torch/nn/modules/module.p

❌ Error processing ctx_ip: Command '['python', 'DeepSEM-master/main.py', '--task', 'non_celltype_GRN', '--data_file', 'deepsem_input_ctx_ip.csv', '--save_name', 'DeepSEM_Results/DeepSEM_out_ctx_ip', '--setting', 'test', '--n_epochs', '120']' returned non-zero exit status 1.

All cell types processed!
